# Rerank existing text retrieval results with Nemotron 3.5

This is the small-fixture starting point: it uses a few synthetic text chunks representing results from an existing retriever, reranks them with `nvidia/Nemotron-3.5-Rerank-8B-BF16`, and measures the change in ranking quality. It does not require the NeMo Retriever Library, an index, or an ingestion pipeline. An optional second example uses Nemotron 3 embeddings to produce the candidates before applying the same reranker. The computation is small, but the first run must download the 8B model and requires a suitable GPU. For a full, much longer-running text-only evaluation, see [Rerank ViDoRe v3 HR](reranking_vidore_v3_hr_text_only.ipynb).

A reranker only reorders the candidates it receives. Retrieve more candidates than you ultimately return so the reranker has useful choices, and evaluate the candidate recall separately.

## Configure local inference

This notebook runs [Nemotron 3.5 Rerank 8B BF16](https://huggingface.co/nvidia/Nemotron-3.5-Rerank-8B-BF16) locally; no API key is required for a public download. If your environment requires authenticated downloads, export `HF_TOKEN` before starting Jupyter. For a token stored in a local `hf-token` file, run `export HF_TOKEN="$(< /path/to/hf-token)"` with the file's actual path. Never paste a token into a notebook cell or commit the file. Only text query/passage pairs are sent to the local model. The loader uses model-provided Python code (`trust_remote_code=True`) at a pinned revision; review that code before running it in a sensitive environment.

Set `RUN_EMBEDDING_RETRIEVAL=False` when you only want the reranker-first example.

In [1]:
RUN_EMBEDDING_RETRIEVAL = False  # Optional, slower embedding path
TOP_K = 5
RERANK_BATCH_SIZE = 1
RERANK_MAX_LENGTH = 512

RERANK_MODEL = "nvidia/Nemotron-3.5-Rerank-8B-BF16"
RERANK_REVISION = "bf773462a362d94d118987c66c57b56f673d5f58"
EMBED_MODEL = "nvidia/Nemotron-3-Embed-1B-BF16"

assert TOP_K > 0
assert RERANK_BATCH_SIZE > 0 and RERANK_MAX_LENGTH > 0

## Dependencies

Local reranking needs a CUDA GPU with BF16 support, working NVIDIA drivers, and enough memory for roughly 16 GiB of weights plus inference overhead. The optional installation cell below does nothing unless you uncomment a `%pip` command.

Install matching PyTorch and torchvision builds for your CUDA environment, then Transformers. The reranker's checkpoint metadata identifies `transformers==5.16.1`; use at least that version for its custom `AutoProcessor` and sequence-classification model. The reranker does **not** need Sentence Transformers. If you also run the optional [Nemotron 3 Embed](https://huggingface.co/nvidia/Nemotron-3-Embed-1B-BF16) section, install `sentence-transformers>=5.4.1`. A clean kernel environment avoids mixing system and user-site packages. Use the [PyTorch installation selector](https://pytorch.org/get-started/locally/) for matching builds. If you change packages, restart the kernel. A missing `ipywidgets` progress bar is only a warning; `operator torchvision::nms does not exist` indicates incompatible PyTorch and torchvision builds.

The first opt-in repair command is for **CUDA 12.4 only**, matching a `torch==2.5.1+cu124` installation. Uncomment it only if that is your intended CUDA wheel; otherwise choose a matching command from [PyTorch's version-specific instructions](https://docs.pytorch.org/get-started/previous-versions/). The second command installs Transformers for reranking. Uncomment the third command only for optional embeddings. If a pre-existing old `datasets` installation fails to import `HfFolder` from `huggingface_hub`, use the fourth command to upgrade `datasets`; the reranker does not require it. `%pip` installs into the selected notebook kernel, not the Jupyter server. Restart the kernel after running any command and rerun the notebook from the top. Installing packages cannot provide a missing GPU or NVIDIA driver; if an incompatibility persists, use a clean GPU-enabled kernel instead.

In [2]:
# Optional CUDA 12.4 repair: uncomment the next line only when needed.
#%pip install "torch==2.5.1" "torchvision==0.20.1" --index-url https://download.pytorch.org/whl/cu124

# Local Nemotron 3.5 reranker (also satisfies the optional embedder's minimum).
#%pip install --upgrade "transformers>=5.16.1,<6"

# Optional local Nemotron 3 Embed dependency.
#%pip install --upgrade "sentence-transformers>=5.4.1"

# Optional repair for an older installed datasets package importing removed HfFolder.
#%pip install --upgrade "datasets>=4.8.2,<5"

In [3]:
from importlib.util import find_spec

required = ("torch", "transformers")
missing = [package for package in required if find_spec(package) is None]
if missing:
    raise ImportError(f"Local reranking requires {', '.join(missing)}. See Dependencies above.")
from importlib.metadata import version
from packaging.version import Version

transformers_version = version("transformers")
if Version(transformers_version) < Version("5.16.1"):
    raise RuntimeError(
        f"Nemotron 3.5 reranking needs transformers>=5.16.1; found {transformers_version}. "
        "Run the optional Transformers %pip command, then restart the kernel."
    )
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Local reranking requires a CUDA GPU visible to this kernel.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("The 8B BF16 reranker requires a CUDA GPU with BF16 support.")

## Prepare imports

Model downloads work without a token when the repository is public. If `HF_TOKEN` is present in the environment, the loader uses it; inference remains local.

In [4]:
import math
import os

from IPython.display import Markdown, display


def show_table(rows, columns):
    def cell(value):
        text = f"{value:.3f}" if isinstance(value, float) else str(value)
        return text.replace("|", "\\|").replace("\n", " ")

    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    lines.extend(
        "| " + " | ".join(cell(row[column]) for column in columns) + " |"
        for row in rows
    )
    display(Markdown("\n".join(lines)))


hf_token = os.environ.get("HF_TOKEN") or None

## Start with post-retrieval candidates

The following fixture represents chunks returned by another retriever. Each candidate retains its original score and rank. The `relevance` labels are only for evaluating this worked example; production retriever output does not need them. Replace each `candidates` list with results from your own retriever.

The examples intentionally place strong answers below weaker lexical matches so the before-and-after comparison is visible.

In [5]:
evaluation_examples = [
    {
        "query": "Can I get a refund if I cancel a flight within 24 hours?",
        "candidates": [
            {
                "chunk_id": "travel-insurance",
                "text": (
                    "Travel insurance can reimburse covered cancellations after "
                    "the insurer reviews a claim."
                ),
                "retrieval_score": 0.94,
                "relevance": 0,
            },
            {
                "chunk_id": "flight-changes",
                "text": (
                    "Passengers can change many tickets before departure, subject "
                    "to fare differences and change fees."
                ),
                "retrieval_score": 0.91,
                "relevance": 1,
            },
            {
                "chunk_id": "twenty-four-hour-rule",
                "text": (
                    "For flights booked at least seven days before departure, you can "
                    "cancel within 24 hours of booking for a full refund."
                ),
                "retrieval_score": 0.88,
                "relevance": 3,
            },
            {
                "chunk_id": "nonrefundable-ticket",
                "text": (
                    "Nonrefundable tickets normally return no cash after the grace "
                    "period, although the airline can issue a travel credit."
                ),
                "retrieval_score": 0.84,
                "relevance": 2,
            },
            {
                "chunk_id": "baggage-refund",
                "text": "Baggage fees are refunded when the airline loses a checked bag.",
                "retrieval_score": 0.81,
                "relevance": 0,
            },
        ],
    },
    {
        "query": "What should I do if my laptop battery is swelling?",
        "candidates": [
            {
                "chunk_id": "battery-life",
                "text": "Lower screen brightness to extend laptop battery life.",
                "retrieval_score": 0.95,
                "relevance": 0,
            },
            {
                "chunk_id": "battery-calibration",
                "text": "Battery calibration can improve an inaccurate charge estimate.",
                "retrieval_score": 0.90,
                "relevance": 0,
            },
            {
                "chunk_id": "swollen-battery-safety",
                "text": (
                    "Stop using and charging a device with a swollen lithium-ion "
                    "battery. Power it down and arrange safe professional removal."
                ),
                "retrieval_score": 0.86,
                "relevance": 3,
            },
            {
                "chunk_id": "battery-disposal",
                "text": (
                    "Take damaged lithium-ion batteries to an approved recycling or "
                    "hazardous-waste facility; do not put them in household trash."
                ),
                "retrieval_score": 0.82,
                "relevance": 2,
            },
            {
                "chunk_id": "laptop-cleaning",
                "text": "Use compressed air to remove dust from laptop vents.",
                "retrieval_score": 0.78,
                "relevance": 0,
            },
        ],
    },
]

for example in evaluation_examples:
    for rank, candidate in enumerate(example["candidates"], start=1):
        candidate["original_rank"] = rank

## Rerank the candidates

`rerank_candidates` accepts ordinary result dictionaries and preserves their metadata. It prepares each text-only `question`/`doc_text` pair with the model's `AutoProcessor`, then reads the sequence-classification logit. The example uses a 512-token cap and batches one passage at a time to limit GPU memory use; adjust `RERANK_MAX_LENGTH` and `RERANK_BATCH_SIZE` to fit your GPU and chunk lengths. It adds `_rerank_score` and sorts by that score. If an upstream stage already added `_rerank_score`, the adapter preserves it as `previous_reranker_score`.

In [6]:
from transformers import AutoModelForSequenceClassification, AutoProcessor

local_processor = AutoProcessor.from_pretrained(
    RERANK_MODEL, revision=RERANK_REVISION, token=hf_token, trust_remote_code=True,
    rerank_max_length=RERANK_MAX_LENGTH,
)
local_reranker = AutoModelForSequenceClassification.from_pretrained(
    RERANK_MODEL, revision=RERANK_REVISION, token=hf_token,
    trust_remote_code=True, dtype=torch.bfloat16,
    attn_implementation="sdpa",
).eval().to("cuda")


def local_rerank_scores(query, passages):
    scores = []
    for start in range(0, len(passages), RERANK_BATCH_SIZE):
        batch = passages[start : start + RERANK_BATCH_SIZE]
        examples = [
            {"question": query, "doc_text": passage, "doc_image": ""}
            for passage in batch
        ]
        inputs = local_processor.process_queries_documents_crossencoder(examples)
        inputs = {
            name: value.to("cuda") if isinstance(value, torch.Tensor) else value
            for name, value in inputs.items()
        }
        with torch.inference_mode():
            logits = local_reranker(**inputs, return_dict=True).logits
        scores.extend(float(score) for score in logits.view(-1).cpu().tolist())
    return scores


def rerank_candidates(query, candidates, top_n=None):
    if not candidates:
        return []
    hits = []
    for candidate in candidates:
        hit = dict(candidate)
        if "_rerank_score" in hit:
            hit["previous_reranker_score"] = hit.pop("_rerank_score")
        hits.append(hit)
    passages = [str(hit["text"]) for hit in hits]
    scores = local_rerank_scores(query, passages)
    if len(scores) != len(hits) or any(not math.isfinite(score) for score in scores):
        raise ValueError("Reranker must return one finite score per candidate.")
    for hit, score in zip(hits, scores):
        hit["_rerank_score"] = score
    ranked = sorted(hits, key=lambda hit: hit["_rerank_score"], reverse=True)
    return ranked[:top_n] if top_n is not None else ranked

/home/edwardk/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|████████████████████████████████████████| 531/531 [00:00<00:00, 3399.07it/s]


In [7]:
rankings = []
for example in evaluation_examples:
    original = [dict(candidate) for candidate in example["candidates"]]
    reranked = rerank_candidates(example["query"], original)
    rankings.append(
        {
            "query": example["query"],
            "original": original,
            "reranked": reranked,
        }
    )

for result in rankings:
    comparison = [
        {
            "reranked_rank": rank,
            "original_rank": hit["original_rank"],
            "chunk_id": hit["chunk_id"],
            "retrieval_score": hit["retrieval_score"],
            "reranker_score": hit["_rerank_score"],
            "relevance": hit["relevance"],
            "text": hit["text"],
        }
        for rank, hit in enumerate(result["reranked"], start=1)
    ]
    print(result["query"])
    show_table(comparison, list(comparison[0]))

Can I get a refund if I cancel a flight within 24 hours?


| reranked_rank | original_rank | chunk_id | retrieval_score | reranker_score | relevance | text |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 3 | twenty-four-hour-rule | 0.880 | -74.214 | 3 | For flights booked at least seven days before departure, you can cancel within 24 hours of booking for a full refund. |
| 2 | 4 | nonrefundable-ticket | 0.840 | -94.653 | 2 | Nonrefundable tickets normally return no cash after the grace period, although the airline can issue a travel credit. |
| 3 | 1 | travel-insurance | 0.940 | -100.376 | 0 | Travel insurance can reimburse covered cancellations after the insurer reviews a claim. |
| 4 | 2 | flight-changes | 0.910 | -101.269 | 1 | Passengers can change many tickets before departure, subject to fare differences and change fees. |
| 5 | 5 | baggage-refund | 0.810 | -104.868 | 0 | Baggage fees are refunded when the airline loses a checked bag. |

What should I do if my laptop battery is swelling?


| reranked_rank | original_rank | chunk_id | retrieval_score | reranker_score | relevance | text |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 3 | swollen-battery-safety | 0.860 | -83.893 | 3 | Stop using and charging a device with a swollen lithium-ion battery. Power it down and arrange safe professional removal. |
| 2 | 4 | battery-disposal | 0.820 | -95.961 | 2 | Take damaged lithium-ion batteries to an approved recycling or hazardous-waste facility; do not put them in household trash. |
| 3 | 1 | battery-life | 0.950 | -105.652 | 0 | Lower screen brightness to extend laptop battery life. |
| 4 | 5 | laptop-cleaning | 0.780 | -106.509 | 0 | Use compressed air to remove dust from laptop vents. |
| 5 | 2 | battery-calibration | 0.900 | -108.010 | 0 | Battery calibration can improve an inaccurate charge estimate. |

## Measure ranking quality

Normalized Discounted Cumulative Gain (nDCG) rewards relevant chunks more when they appear near the top. The graded labels in this fixture range from `0` for irrelevant to `3` for a direct answer. A positive delta means the reranker improved the ordering for these candidates.

Reranking does not change candidate recall because it cannot add a missing chunk.

In [8]:
def ndcg_at_k(hits, k):
    relevances = [float(hit["relevance"]) for hit in hits[:k]]
    dcg = sum(
        (2**relevance - 1) / math.log2(rank + 1)
        for rank, relevance in enumerate(relevances, start=1)
    )
    ideal = sorted(
        (float(hit["relevance"]) for hit in hits),
        reverse=True,
    )[:k]
    idcg = sum(
        (2**relevance - 1) / math.log2(rank + 1)
        for rank, relevance in enumerate(ideal, start=1)
    )
    return dcg / idcg if idcg else 0.0


quality_rows = []
for result in rankings:
    before = ndcg_at_k(result["original"], TOP_K)
    after = ndcg_at_k(result["reranked"], TOP_K)
    quality_rows.append(
        {
            "query": result["query"],
            f"nDCG@{TOP_K} before": before,
            f"nDCG@{TOP_K} after": after,
            "delta": after - before,
        }
    )

quality = quality_rows + [
    {
        "query": "Mean",
        f"nDCG@{TOP_K} before": sum(
            row[f"nDCG@{TOP_K} before"] for row in quality_rows
        ) / len(quality_rows),
        f"nDCG@{TOP_K} after": sum(
            row[f"nDCG@{TOP_K} after"] for row in quality_rows
        ) / len(quality_rows),
        "delta": sum(row["delta"] for row in quality_rows) / len(quality_rows),
    }
]
show_table(quality, list(quality[0]))

| query | nDCG@5 before | nDCG@5 after | delta |
| --- | --- | --- | --- |
| Can I get a refund if I cancel a flight within 24 hours? | 0.577 | 0.993 | 0.415 |
| What should I do if my laptop battery is swelling? | 0.539 | 1.000 | 0.461 |
| Mean | 0.558 | 0.996 | 0.438 |

## Add or replace reranking in an existing system

To **add** reranking, pass the ordered dictionaries from your retriever to `rerank_candidates`. Use `top_n` only after reranking.

To **replace** an existing reranker, disable the upstream reranking stage and request a larger raw candidate set from the embedding or lexical retriever. Preserve the chunk text and identifiers when adapting its output. Applying multiple rerankers is possible, but each stage adds cost and later stages cannot recover candidates that an earlier stage removed.

Retriever scores and reranker scores have different meanings. Compare ranks or evaluated quality instead of comparing the numeric scores directly.

## Optional: produce candidates with embeddings

This section retains the embedding path while keeping the reranker as the focus. It embeds the first example's small candidate corpus, performs cosine-similarity retrieval, and sends the top candidates to the same `rerank_candidates` adapter. In an application, replace this in-memory search with results from your existing vector database. Local Nemotron 3 Embed needs `transformers>=5.2.0` and `sentence-transformers>=5.4.1`; the separate opt-in `%pip` commands in Dependencies install them. Sentence Transformers is only used here, not by the reranker. Hosted mode does not need these packages.

In [9]:
def cosine_similarity(left, right):
    dot = sum(a * b for a, b in zip(left, right))
    left_norm = math.sqrt(sum(value * value for value in left))
    right_norm = math.sqrt(sum(value * value for value in right))
    return dot / (left_norm * right_norm)


if RUN_EMBEDDING_RETRIEVAL:
    embedding_example = evaluation_examples[0]
    corpus = [dict(candidate) for candidate in embedding_example["candidates"]]
    documents = [candidate["text"] for candidate in corpus]

    import transformers
    try:
        import sentence_transformers
    except ImportError as exc:
        if "HfFolder" in str(exc) and "huggingface_hub" in str(exc):
            raise RuntimeError(
                "An old datasets package is incompatible with huggingface_hub. "
                "Run the optional datasets %pip command in Dependencies, "
                "then restart the kernel."
            ) from None
        raise

    minimum_versions = {"transformers": "5.2.0", "sentence-transformers": "5.4.1"}
    installed_versions = {
        "transformers": transformers.__version__,
        "sentence-transformers": sentence_transformers.__version__,
    }
    outdated = [
        f"{name}=={installed_versions[name]} (need >={minimum})"
        for name, minimum in minimum_versions.items()
        if Version(installed_versions[name]) < Version(minimum)
    ]
    if outdated:
        raise RuntimeError(
            "Local Nemotron 3 embedding dependencies are too old: "
            + ", ".join(outdated)
            + ". Run the optional Transformers and Sentence Transformers "
            "%pip commands in Dependencies, then restart the kernel."
        )

    from sentence_transformers import SentenceTransformer

    embedder = SentenceTransformer(EMBED_MODEL, device="cuda")
    document_vectors = embedder.encode_document(documents).tolist()
    query_vector = embedder.encode_query([embedding_example["query"]])[0].tolist()
    del embedder

    similarities = [
        cosine_similarity(query_vector, vector) for vector in document_vectors
    ]
    candidate_order = sorted(
        range(len(corpus)), key=lambda index: similarities[index], reverse=True
    )[:TOP_K]
    embedding_candidates = []
    for original_rank, index in enumerate(candidate_order, start=1):
        candidate = dict(corpus[index])
        candidate["retrieval_score"] = similarities[index]
        candidate["original_rank"] = original_rank
        embedding_candidates.append(candidate)

    embedding_reranked = rerank_candidates(
        embedding_example["query"],
        embedding_candidates,
        top_n=TOP_K,
    )
    embedding_quality = [
        {
            "ranking": "Embedding retrieval",
            f"nDCG@{TOP_K}": ndcg_at_k(embedding_candidates, TOP_K),
        },
        {
            "ranking": "Embedding retrieval + reranking",
            f"nDCG@{TOP_K}": ndcg_at_k(embedding_reranked, TOP_K),
        },
    ]
    show_table(embedding_quality, list(embedding_quality[0]))
    show_table(
        embedding_reranked,
        ["original_rank", "chunk_id", "retrieval_score", "_rerank_score", "relevance", "text"],
    )
else:
    print("Embedding retrieval skipped. Set RUN_EMBEDDING_RETRIEVAL=True to run it.")

Embedding retrieval skipped. Set RUN_EMBEDDING_RETRIEVAL=True to run it.


## Tradeoffs when you add a reranker

- **Quality and recall:** A cross-encoder can improve candidate ordering, but it cannot recover relevant chunks that the first-stage retriever did not return. Measure first-stage recall and reranked nDCG separately.
- **Candidate depth:** More candidates give the reranker more opportunities to find the best chunks, but increase inference cost. Tune the retrieval depth and final `top_n` together.
- **Latency and throughput:** A reranker scores every query-candidate pair. Batching improves throughput, while smaller candidate sets reduce latency.
- **Input length:** Long chunks can be truncated. Align chunk size and the model's input limit so the evidence that determines relevance is retained.
- **Score interpretation:** Reranker scores order candidates for one query; they are not calibrated probabilities and should not be compared across queries without validation.
- **Model fit:** Evaluate the reranker on your domains and languages. This example uses text chunks only.
- **Deployment:** Local inference keeps query and passage text in your environment, but requires a BF16-capable GPU, significant model storage and memory, and operational capacity. An optional Hugging Face token is used only for model download.